### This script creates embeddings for each embedding model for each dataset.
Embeddings are saved in the Embeddings folder within the Datasets folder
Note that the script is currently hard-coded for IISE 2024

In [2]:
# Importing packages
import pandas as pd
from sentence_transformers import SentenceTransformer
import pickle
from nltk.tokenize import sent_tokenize, word_tokenize
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import numpy as np
import pickle

c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [ ]:
#Configuration (environment variables)
iise_dataset_filepath = './Datasets/IISE 2024/IISE2024_Processed.csv'
informs_dataset_filepath = r'.\Datasets\INFORMS 2024\INFORMS2024_processed.csv'
embeddings_save_filepath = ''
lemmatized_dataset_save_filepath = ''

class EmbModel:
    def __init__(self, name, doChunk):
        self.name = name
        self.doChunk = doChunk
        self.embedding = None  # Initialize the embedding attribute to None

    def set_embedding(self, embedding_array):
        """Sets the embedding attribute to the provided ndarray."""
        if isinstance(embedding_array, np.ndarray):
            self.embedding = embedding_array
        else:
            raise TypeError("embedding must be a numpy ndarray.")

emb_lst_base = []
emb_lst_base.append(EmbModel(name='Snowflake/snowflake-arctic-embed-m-v1.5',doChunk=False))
emb_lst_base.append(EmbModel(name='ibm-granite/granite-embedding-125m-english',doChunk=False))
emb_lst_base.append(EmbModel(name='Snowflake/snowflake-arctic-embed-s',doChunk=False))
emb_lst_base.append(EmbModel(name='BAAI/bge-small-en-v1.5',doChunk=False))
emb_lst_base.append(EmbModel(name='sentence-transformers/all-MiniLM-L12-v2',doChunk=True))
emb_lst_base.append(EmbModel(name='sentence-transformers/all-MiniLM-L6-v2',doChunk=True))

print(emb_lst_base)

[<__main__.EmbModel object at 0x00000164E4916F30>, <__main__.EmbModel object at 0x00000164D3996030>, <__main__.EmbModel object at 0x00000164D39943E0>, <__main__.EmbModel object at 0x00000164D39966C0>, <__main__.EmbModel object at 0x00000164D39970B0>, <__main__.EmbModel object at 0x00000164E4E85370>]


In [9]:
df = pd.read_csv(informs_dataset_filepath)
df.head(5)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5525 entries, 0 to 5524
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   paper_id      5525 non-null   int64 
 1   session_name  5525 non-null   object
 2   paper_title   5525 non-null   object
 3   abstract      5525 non-null   object
 4   session_id    5525 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 215.9+ KB


In [10]:
def create_sentence_dataframe(df):
    """
    Creates a new DataFrame from an input DataFrame, splitting abstracts into sentences.

    Args:
        df: The input DataFrame with 'paper_id', 'abstract', and 'paper_title' columns.

    Returns:
        A new DataFrame with 'paper_id', 'sentence', and 'source' columns, or 
        an empty DataFrame if the input DataFrame is empty or doesn't have the 
        required columns.
    """

    if df.empty:
      return pd.DataFrame() #Return empty df if input is empty

    required_cols = ['paper_id', 'abstract', 'paper_title']
    if not all(col in df.columns for col in required_cols):
        print("Error: Input DataFrame must contain columns 'paper_id', 'abstract', and 'paper_title'.")
        return pd.DataFrame() # Return empty if columns are missing

    new_rows = []
    for index, row in df.iterrows():
        paper_id = row['paper_id']
        abstract = row['abstract']
        title = row['paper_title']

        new_rows.append({'paper_id': paper_id,
                         'sentence': title,
                         'source': "Title"})
        try:
          sentences = sent_tokenize(abstract)
        except:
           print(abstract)
        for sentence in sentences:
            new_rows.append({'paper_id': paper_id,
                             'sentence': sentence,
                             'source': "Abstract"})

    new_df = pd.DataFrame(new_rows)
    return new_df
    
def split_into_sentences(df, col_name):
  """
  Splits a DataFrame with a single row containing a body of text into 
  multiple rows, one per sentence.

  Args:
    df: A pandas DataFrame with a single row and a single column containing text.

  Returns:
    A new pandas DataFrame with one row per sentence.
  """
  text = df.iloc[0, 0]  # Extract the text from the DataFrame
  sentences = nltk.sent_tokenize(text)  # Split the text into sentences
  return pd.DataFrame(sentences, columns=[col_name])

#Sentenizing data
chunked_df = create_sentence_dataframe(df)

#Concatenating title into abstract for non-chunk models
non_chunked_df = df.copy()
non_chunked_df['text'] = non_chunked_df['paper_title'] + '. ' + non_chunked_df['abstract']
non_chunked_df = non_chunked_df[['paper_id','text']]

display(non_chunked_df.head(5))
chunked_df.head(20)

,paper_id,text
0,0,Quantum Optimization. Quantum computing has go...
1,1,GAMSPy and Data APIs for Streamlining Optimiza...
2,2,Elevate your Optimization Practice with latest...
3,3,Analyzing Multidimensional Data and building P...
4,4,"HEXALY, A New Kind of Global Optimization Solv..."


,paper_id,sentence,source
0,0,Quantum Optimization,Title
1,0,Quantum computing has gone from the lab to the...,Abstract
2,0,While you may think production use of quantum ...,Abstract
3,0,This workshop will cover the following topics:...,Abstract
4,0,This will include live demos and an introducti...,Abstract
5,0,Leap provides access to a portfolio of hybrid ...,Abstract
6,0,How to get started with quantum for business.,Abstract
7,1,GAMSPy and Data APIs for Streamlining Optimiza...,Title
8,1,GAMS (General Algebraic Modeling System) is an...,Abstract
9,1,"However, as optimization becomes an integrated...",Abstract


In [ ]:
display(chunked_df.info())
display(non_chunked_df.info())
#generating embeddings

file_prefix = r'.\Datasets\IISE 2024\Embeddings'

for i in range(0,len(emb_lst_base)):

    #Getting file name
    model_name = emb_lst_base[i].name
    print(model_name)
    file_suffix = '\\' + model_name.split('/')[-1] + '.pkl'
    filename = file_prefix + file_suffix

    #Getting embedding and encoding
    emb_name = emb_lst_base[i].name
    embedding_model = SentenceTransformer(emb_name)
    if emb_lst_base[i].doChunk:
        embeddings = embedding_model.encode(chunked_df['sentence'], show_progress_bar=True)
    else:
        embeddings = embedding_model.encode(non_chunked_df['text'], show_progress_bar=True)
    emb_lst_base[i].set_embedding(embeddings)
    print(len(embeddings))

    #Saving embeddings
    with open (filename, 'wb') as f:
      pickle.dump(emb_lst_base[i], f)

    #Deleting embeddings after save
    del embedding_model
    emb_lst_base[i] = 0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7995 entries, 0 to 7994
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   paper_id  7995 non-null   int64 
 1   sentence  7995 non-null   object
 2   source    7995 non-null   object
dtypes: int64(1), object(2)
memory usage: 187.5+ KB


None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 850 entries, 0 to 849
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   paper_id  850 non-null    int64 
 1   text      850 non-null    object
dtypes: int64(1), object(1)
memory usage: 13.4+ KB


None

Snowflake/snowflake-arctic-embed-m-v1.5


...

850
sentence-transformers/all-MiniLM-L12-v2


c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Batches: 100%|██████████| 250/250 [01:41<00:00,  2.46it/s]


7995
sentence-transformers/all-MiniLM-L6-v2


Batches: 100%|██████████| 250/250 [00:54<00:00,  4.56it/s]


7995


In [13]:
for i in range(0,len(emb_lst_base)):
    print(emb_lst_base[i].name)
    print(len(emb_lst_base[i].embedding))

AttributeError: 'int' object has no attribute 'name'

In [ ]:
filename = r'.\Datasets\INFORMS 2024\chunked_df.csv'
filename_2 = r'.\Datasets\INFORMS 2024\non_chunked_df.csv'

chunked_df.to_csv(filename)
non_chunked_df.to_csv(filename_2)

# Testing various ensemble models
Ensuring they work

Assumptions:
*Retrieval is most relevant task over STS despite differences in dataset size
*Queries are assumed to have results in the dataset that are relevant to the query

Selection procedure:
0: Eng V2
1: Filter models for open source, sentenceTransformers compatible
2: 150M parameters or less
3: Token size greater or equal to 512 (to capture entire dataset)
4: Sort by descending retrieval scores
5: Selected two best performing models greater than 100M parameters (yet still less than 150M) and two best performing models less than 100M parameters (with the hypothesis that these may be faster) while not taking multiple versions of the same model (e.g. only one snowflake model per pair)
6: Selected all-MiniLM-L12-v2 and all-MiniLM-L6-v2 as additional models with sentenceization

Metrics:
- Average retrieval time
- Retrieval score (as determined by 'judge' LLM 1 OR by comparision with BERTopic clustering)
- OPTIONAL: Retrieval score (as determined by 'judge' LLM 2)
- Reranking score (compared to model with high reranking MTEB score 1)
- Reranking score (compared to model with high reranking MTEB score 2)

First dataset: IISE 2024
Second dataset: INFORMS

Notes:
Generate sample queries with LLM
Queries should include multiple query types
I should not lemmatize text as this may be detrimental. I need to find sources to support this.
Note in paper that for both INFORMS and IISE 2024, null (and values such as abstract: 'tbd') values are removed

### Verifing embedding model works with sentence transformers

### snowflake-arctic-embed-m-v1.5

In [2]:
test_string = "Pickleball spikeball basketball football soccer"

model_ID = "Snowflake/snowflake-arctic-embed-m-v1.5"
model = SentenceTransformer(model_ID)
document_embedding = model.encode(test_string)
print(document_embedding)

c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tillman\.cache\huggingface\hub\models--Snowflake--snowflake-arctic-embed-m-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[ 2.55715032e-03 -2.45518871e-02  7.06477985e-02 -2.37865914e-02
  7.19385222e-02  1.82218738e-02  1.50122168e-02  8.30065683e-02
 -8.86021033e-02 -1.03426374e-01 -5.34168771e-03  2.97219492e-02
 -3.93591858e-02  8.20738543e-03  4.81614955e-02  4.97263372e-02
  3.53955291e-02  4.02024090e-02  1.18773170e-01 -1.50480745e-02
 -8.79301503e-02  6.19610120e-03  6.51864260e-02 -2.39311047e-02
  4.18231562e-02  1.79127473e-02 -1.35076046e-02  3.21577750e-02
 -5.48595190e-02  2.46448424e-02 -3.32119502e-02  4.18611150e-03
  1.02317236e-01 -4.60151657e-02 -2.04614419e-02 -9.99837648e-03
  3.50894853e-02  3.62702757e-02 -5.67618757e-03 -7.20288530e-02
 -8.25406983e-03 -6.02377653e-02 -1.63383298e-02  3.59887220e-02
  6.74927142e-03 -4.89413366e-03 -2.56796032e-01  7.45391995e-02
 -3.77975516e-02 -7.53035620e-02 -8.84273052e-02  4.75622043e-02
 -1.12922722e-02 -1.86251872e-03 -1.97144132e-02  6.18393009e-04
 -2.31212890e-03 -3.51458192e-02  1.05535900e-02 -4.84653190e-02
  4.25197631e-02  3.28622

### ibm-granite/granite-embedding-125m-english

In [3]:
test_string = "Pickleball spikeball basketball football soccer"

model_ID = "ibm-granite/granite-embedding-125m-english"
model = SentenceTransformer(model_ID)
document_embedding = model.encode(test_string)
print(document_embedding)

c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tillman\.cache\huggingface\hub\models--ibm-granite--granite-embedding-125m-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[-1.48671512e-02 -4.48924899e-02 -3.60993631e-02 -5.54561019e-02
  1.79914688e-03 -6.03393726e-02  2.30351239e-02 -1.90064423e-02
  1.80597696e-03  2.99249422e-02 -3.86953652e-02  3.64606492e-02
 -2.93862708e-02 -4.58758064e-02 -7.97588192e-03 -9.21037234e-03
 -5.01196533e-02  4.77301255e-02  6.35408312e-02 -3.86843830e-02
  4.60931426e-03 -4.96120192e-02 -2.37487014e-02 -4.81372438e-02
  3.40782315e-03 -3.36845443e-02  3.94622982e-02  8.37885309e-03
 -2.36585550e-02 -7.85061129e-05 -4.24300246e-02  1.84985343e-02
 -1.15210777e-02 -1.55791594e-03  4.64887023e-02 -1.85944699e-02
  2.64649596e-02 -5.26948483e-04  6.18280983e-03 -2.19951048e-02
  2.13001156e-03  9.12666321e-03 -4.72346097e-02 -8.93126521e-03
 -8.44120421e-03  2.10954957e-02  1.80008658e-03 -2.58251559e-04
 -1.08544333e-02  5.31299077e-02  4.80179042e-02 -2.62050871e-02
  5.42604830e-03 -1.79907046e-02  2.56937072e-02  2.36030482e-02
 -4.59997542e-03 -2.30999626e-02 -8.66051111e-03  2.60021677e-03
  4.32111323e-03  2.44989

### snowflake-arctic-embed-s

In [4]:
test_string = "Pickleball spikeball basketball football soccer"

model_ID = "Snowflake/snowflake-arctic-embed-s"
model = SentenceTransformer(model_ID)
document_embedding = model.encode(test_string)
print(document_embedding)

No sentence-transformers model found with name Snowflake/snowflake-arctic-embed-s. Creating a new one with mean pooling.
c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tillman\.cache\huggingface\hub\models--Snowflake--snowflake-arctic-embed-s. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-

[-2.82215655e-01 -1.24881551e-01 -2.89380521e-01  4.13412243e-01
  7.34822899e-02  2.77441554e-02 -1.92491487e-01 -6.27084523e-02
  4.23421487e-02  2.13638693e-01 -2.45881125e-01  9.63115715e-04
  1.73329145e-01 -5.76113582e-01  4.39105220e-02 -1.11417674e-01
 -3.56310248e-01 -3.51959288e-01  5.14897332e-02  2.57156253e-01
 -4.24100965e-01 -7.35390544e-01  3.98268878e-01  5.74839890e-01
 -2.72142775e-02 -7.69741684e-02  5.88406734e-02  4.94760126e-01
  5.11932254e-01  1.22183338e-01 -2.02125862e-01 -2.46599481e-01
 -5.71310930e-02  8.19815919e-02 -2.08034188e-01 -2.47302935e-01
 -7.75508732e-02  2.01484710e-01  6.70589656e-02  2.01322436e-02
 -1.30183399e-01 -2.98989117e-01  8.72626156e-02 -3.52101028e-02
  3.11754107e-01  3.36889662e-02  3.38967234e-01 -3.71477127e-01
  2.25281358e-01  4.98381764e-01  1.14173844e-01 -1.52645707e-02
  1.07148848e-01  1.34319037e-01 -4.22748365e-03 -5.38747191e-01
 -4.12469864e-01 -1.66517258e-01 -2.02571839e-01 -1.33465707e-01
  6.68563008e-01 -3.30552

### snowflake-arctic-embed-m-v1.5

In [5]:
test_string = "Pickleball spikeball basketball football soccer"

model_ID = "BAAI/bge-small-en-v1.5"
model = SentenceTransformer(model_ID)
document_embedding = model.encode(test_string)
print(document_embedding)

c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tillman\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[-0.0288029  -0.03289248  0.02624095 -0.10924903  0.02854998  0.06101682
  0.00944559  0.04762219  0.01149689  0.00056774 -0.0268832  -0.04978728
 -0.0063849   0.00986785  0.03164564 -0.02872504  0.00490647 -0.03267572
 -0.06762824 -0.00646775  0.00756815 -0.02502199 -0.0198386  -0.04122998
 -0.00606799 -0.00604843  0.02189269 -0.00346001 -0.0741123  -0.10050332
  0.05953201  0.02361215 -0.01557252 -0.02141573  0.00215384 -0.01896626
 -0.00934666 -0.02032896 -0.04137335  0.01415776 -0.0376231  -0.00093058
  0.03592693 -0.06550256 -0.02042165 -0.03867671 -0.02599833  0.00078899
  0.10822343 -0.03022554 -0.02923823  0.01779851 -0.0091145   0.01069591
  0.02453842  0.06034641  0.03919393  0.00807073  0.04762332  0.00328627
  0.01571994  0.04640777 -0.15192948  0.09106022 -0.0421855  -0.02342782
 -0.01110566  0.00948537  0.03293406  0.03791908  0.01766115  0.01992568
  0.04297939  0.06892051  0.00236617  0.0499365   0.029782   -0.08239006
 -0.03072623  0.00789585 -0.05094376 -0.00739595 -0

### all-MiniLM-L12-v2

In [7]:
test_string = "Pickleball spikeball basketball football soccer"

model_ID = "sentence-transformers/all-MiniLM-L12-v2"
model = SentenceTransformer(model_ID)
document_embedding = model.encode(test_string)
print(document_embedding)

c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tillman\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Tillman\anaconda3\envs\ThesisModel\Lib\site-packages\transformers\toke

[-3.70959640e-02  4.65521663e-02  4.23599780e-02 -8.68050084e-02
  2.17017550e-02  2.89835483e-02  1.26859531e-01 -3.57786864e-02
 -4.43361253e-02  2.99790297e-02 -4.91277091e-02 -3.05560045e-02
 -6.32948130e-02  4.86108474e-02  4.60534133e-02  3.57258413e-03
  3.85361686e-02 -5.32639995e-02 -2.38597225e-02 -1.37966676e-02
  1.15557835e-02  2.61888895e-02 -2.58655511e-02 -5.27340285e-02
 -4.83747497e-02 -1.20546045e-02 -3.96453775e-02  6.44857064e-02
 -2.21780464e-01 -8.74493197e-02  6.62420224e-03  3.63956615e-02
  5.33894412e-02  1.95575834e-04  1.60565618e-02  9.52685659e-04
  3.31632160e-02  2.97715683e-02 -4.40547019e-02  1.03791384e-02
 -7.55669316e-03 -1.23958223e-01  7.73789212e-02 -1.23527078e-02
 -4.33653668e-02  4.86014523e-02  2.22266801e-02  2.15617903e-02
  1.92129165e-02  2.67937891e-02  4.39928472e-02  1.03068858e-01
 -1.54360151e-02  6.35101646e-02  7.32755214e-02  1.07833613e-02
  2.55919695e-02  2.82539371e-02  1.90508757e-02  4.21233661e-02
 -3.34517322e-02  3.40741

### all-MiniLM-L6-v2

In [ ]:
test_string = "Pickleball spikeball basketball football soccer"

model_ID = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_ID)
document_embedding = model.encode(test_string)
print(document_embedding)

[-5.23509160e-02 -1.51457814e-02  2.22264975e-02 -5.67874797e-02
 -3.10996044e-02 -9.33729019e-03  8.59456062e-02 -1.04952296e-02
  6.03465252e-02  4.07492146e-02 -7.81145543e-02 -3.19846682e-02
 -2.05064192e-02  2.96153743e-02  1.67275779e-02 -2.02961895e-03
 -7.23551586e-03 -4.19696979e-02 -1.86338406e-02 -6.72782063e-02
 -6.32193461e-02 -4.59876843e-02 -2.44936935e-04  7.48707587e-03
  5.84188569e-03  5.84038459e-02  8.37186575e-02  1.13228709e-01
 -1.57078043e-01 -1.25050917e-01 -2.76275240e-02  3.11880326e-03
 -4.68062013e-02  2.99202725e-02 -6.27445476e-03 -1.65460464e-02
 -1.15859807e-02 -3.01920343e-03 -4.62194048e-02  1.33018065e-02
  5.57323545e-03 -9.11694616e-02  3.94974314e-02  7.42195100e-02
 -6.75334036e-02  8.48516077e-02  1.05431490e-03  2.50278506e-03
 -1.27268117e-02 -7.26903882e-03 -4.15814184e-02 -1.14932191e-02
  3.39939296e-02  2.85275374e-02  1.39319897e-01  6.51506260e-02
  3.19519714e-02  3.45373340e-02  1.75315961e-02  3.13136391e-02
  1.39152473e-02  1.04620

#### Deleted Code Dump

In [ ]:
#Cleaning dataset: selecting relevant columns and removing all others

#Creating a dictionary to generalize the column names of a dataset
col_dict = {'paper_id': 'Application #',
            'title': 'Application Name',
            'abstract': 'Abstract Description',
            'filter_attendee_role': 'Who is the primary audience?'
            }

#Filtering dataset for columns given in dictionary values
def rename_and_select_columns(df, col_dict):
    """
    Selects columns in a DataFrame that match values in a dictionary,
    renames them with the dictionary keys, and removes other columns.

    Args:
        df: The input DataFrame.
        col_dict: A dictionary where values are original column names and keys are desired column names.

    Returns:
        A new DataFrame with renamed and selected columns, or the original
        DataFrame if no matching columns are found.  Returns an empty DataFrame
        if the input DataFrame is empty.
    """

    if df.empty:
        return df  # Return the empty DataFrame if input is empty.

    matching_cols = []
    for new_name, original_name in col_dict.items():
        if original_name in df.columns:
            matching_cols.append(original_name)

    if not matching_cols:  # Handle the case where no columns match
        return df  # Or raise an exception, or return an empty DataFrame, as needed.

    new_df = df[matching_cols].copy() # Create a copy to avoid SettingWithCopyWarning

    for new_name, original_name in col_dict.items():
         if original_name in new_df.columns:
             new_df = new_df.rename(columns={original_name: new_name})

    return new_df

In [ ]:
#Getting string size metadata from dataset

iise_dataset_filepath = './Datasets/IISE 2024/IISE2024_Processed.csv'
informs_dataset_filepath = r'.\Datasets\INFORMS 2024\INFORMS2024_processed.csv'

iise_df = pd.read_csv(iise_dataset_filepath)
informs_df = pd.read_csv(informs_dataset_filepath)

iise_df['text'] = iise_df['paper_title'] + '. ' + iise_df['abstract']
informs_df['text'] = informs_df['paper_title'] + '. ' + informs_df['abstract']

def analyze_string_column(df, column_name):
    """
    Analyzes a string column in a Pandas DataFrame and returns the longest, 
    average, and median string lengths, the average number of words,
    and the proportion of strings with more than 256 words.

    Args:
        df: The Pandas DataFrame.
        column_name: The name of the column containing strings.

    Returns:
        A dictionary containing the longest, average, and median string lengths,
        the average number of words, and the proportion of strings with a 
        word count greater than 256.
        Returns an empty dictionary if the column does not exist or if it does not contain strings
    """

    if column_name not in df.columns:
        print(f"Error: Column '{column_name}' not found in DataFrame.")
        return {}

    string_lengths = []
    word_counts = []
    long_string_count = 0 
    for value in df[column_name]:
        if isinstance(value, str):
            string_lengths.append(len(value))
            num_words = len(value.split())
            word_counts.append(num_words)
            if num_words > (384):
                long_string_count += 1
        elif not pd.isna(value):
            print(f"Warning: Non-string value encountered in column '{column_name}'. Ignoring.")

    if not string_lengths:
        print(f"Warning: No valid string values found in column '{column_name}'.")
        return {}

    longest = max(string_lengths)
    average_length = np.mean(string_lengths)
    median_length = np.median(string_lengths)
    average_words = np.mean(word_counts)
    longest_words = max(word_counts)
    median_words = np.median(word_counts)
    proportion_long_strings = long_string_count / len(string_lengths)

    return {
        "longest": longest,
        "average_length": average_length,
        "median_length": median_length,
        "longest_words": longest_words,
        "average_words": average_words,
        "median_words": median_words,
        "% over 384" : proportion_long_strings
    }
print('IISE data:')
print(analyze_string_column(iise_df,'text'))

print('Informs data:')
print(analyze_string_column(informs_df,'text'))

display(informs_df.info())

IISE data:
{'longest': 2138, 'average_length': 1445.3811764705883, 'median_length': 1527.5, 'longest_words': 280, 'average_words': 202.89411764705883, 'median_words': 215.0, '% over 384': 0.0}
Informs data:
{'longest': 3529, 'average_length': 1237.9878733031674, 'median_length': 1257.0, 'longest_words': 490, 'average_words': 174.0952036199095, 'median_words': 176.0, '% over 384': 0.00036199095022624434}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5525 entries, 0 to 5524
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   paper_id      5525 non-null   int64 
 1   session_name  5525 non-null   object
 2   paper_title   5525 non-null   object
 3   abstract      5525 non-null   object
 4   session_id    5525 non-null   int64 
 5   text          5525 non-null   object
dtypes: int64(2), object(4)
memory usage: 259.1+ KB


None